# Quantum Physics - Mathematical Tools Demo Notebook
Based on Incoming and Outgoing Competences

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, logm, fractional_matrix_power

# Helper display
def format_complex(z):
        a = np.round(z.real, 2)
        b = np.round(z.imag, 2)
        if b >= 0:
            return f"{a} + {b}i"
        else:
            return f"{a} - {abs(b)}i"

def print_matrix(M):
    M = np.atleast_2d(M)  # Ensure it's 2D
    if M.size == 1:
        output = format_complex(M[0,0])
    else:
        rows = []
        for row in M:
            rows.append("[ " + " , ".join(format_complex(z) for z in row) + " ]")

        output = "\n".join(rows)
    print(output)



## 1. Scalar product in Cⁿ

In [2]:
# Example vectors in C^3
v = np.array([1+1j, 2, -1j])
w = np.array([2, -1j, 3])

# Scalar product: <v, w> = sum(conj(v_i) * w_i)
scalar_product = np.sum(np.conjugate(v) * w) # By definition
print_matrix(scalar_product)

scalar_product = np.conjugate(v).T @ w # Do it with matrix product @
print_matrix(scalar_product)

scalar_product = np.vdot(v, w)  # vdot does conjugate(v) * w
print_matrix(scalar_product)

2.0 - 1.0i
2.0 - 1.0i
2.0 - 1.0i


## 2. Dyadic product in Cⁿ

In [3]:
v = np.array([1+1j, 2])
w = np.array([2, -1j])

# Dyadic (outer) product: v ⊗ w†
def dyadic_product(v, w):
    M = np.zeros((len(v), len(w)), dtype=complex)
    for i in range(len(v)):
        for j in range(len(w)):
            M[i, j] = v[i] * np.conjugate(w[j])
    return M

dyadic_product = dyadic_product(v, w)
print_matrix(dyadic_product)

print('\n') # Add spacing

dyadic_product = np.outer(v, np.conjugate(w)) # Use built-in function
print_matrix(dyadic_product)

[ 2.0 + 2.0i , -1.0 + 1.0i ]
[ 4.0 + 0.0i , -0.0 + 2.0i ]


[ 2.0 + 2.0i , -1.0 + 1.0i ]
[ 4.0 + 0.0i , -0.0 + 2.0i ]


## 3. Eigenvalue problems

In [4]:
# --- 3x3 Hermitian eigenproblem: characteristic polynomial by hand + eigh comparison ---

import numpy as np

# Example 3x3 Hermitian matrix
H = np.array([[2, 1-1j, 0],
              [1+1j, 3, 2],
              [0, 2, 1]], dtype=complex)

# --- Build characteristic polynomial coefficients by hand ---
# We form the monic polynomial p(λ) = λ^3 - c1 λ^2 + c2 λ - c3,
# where for a 3x3 matrix:
#   c1 = tr(H)
#   c2 = 1/2 ( (tr H)^2 - tr(H^2) )
#   c3 = det(H)

trH = np.trace(H)
trH2 = np.trace(H @ H)
c1 = trH
c2 = 0.5 * (trH**2 - trH2)
c3 = np.linalg.det(H)

coeffs = [1.0, -c1, c2, -c3]   # coefficients for np.roots: [1, -c1, c2, -c3]

# --- Solve polynomial for eigenvalues ---
eigvals_poly = np.roots(coeffs)

# Numerical cleanup: eigenvalues of a Hermitian should be real (allow tiny imag parts)
eigvals_poly = np.real_if_close(eigvals_poly, tol=1e-8)
eigvals_poly_sorted = np.sort(eigvals_poly)

print("Eigenvalues from characteristic polynomial (sorted):  ", np.round(eigvals_poly_sorted, 6))

# --- Numerical solution via eigh (guaranteed real for Hermitian) ---
eigvals_eigh, eigvecs_eigh = np.linalg.eigh(H)
print("\nEigenvalues from np.linalg.eigh (sorted):            ", np.round(eigvals_eigh, 6))

for eigval, eigvec in zip(eigvals_eigh, eigvecs_eigh.T):
    print(f"\nEigenvector corresponding to eigenvalue {eigval}:")
    print_matrix(eigvec)


Eigenvalues from characteristic polynomial (sorted):   [-0.489289  1.710831  4.778457]

Eigenvalues from np.linalg.eigh (sorted):             [-0.489289  1.710831  4.778457]

Eigenvector corresponding to eigenvalue -0.48928857181007895:
[ -0.32 + 0.0i , 0.4 + 0.4i , -0.54 - 0.54i ]

Eigenvector corresponding to eigenvalue 1.710831453551689:
[ -0.85 + 0.0i , 0.12 + 0.12i , 0.35 + 0.35i ]

Eigenvector corresponding to eigenvalue 4.778457118258389:
[ -0.41 + 0.0i , -0.57 - 0.57i , -0.3 - 0.3i ]


## 4. Projectors

In [5]:
# Normalized vector
v = np.array([1, 1j]) / np.sqrt(2)

# Projector P = |v⟩⟨v|
P = np.outer(v, np.conjugate(v))
print_matrix(P)

[ 0.5 + 0.0i , 0.0 - 0.5i ]
[ 0.0 + 0.5i , 0.5 + 0.0i ]


## 5. Unitary matrices and spectral decomposition

In [6]:
U = np.array([[1/np.sqrt(2), 1j/np.sqrt(2)],
              [1j/np.sqrt(2), 1/np.sqrt(2)]])

# Check unitarity: U†U = I
is_unitary = np.allclose(np.eye(2), np.conjugate(U.T) @ U)

# Eigenvalues of a unitary
eigvals, eigvecs = np.linalg.eig(U)

print(is_unitary)
print_matrix(eigvals)
print_matrix(np.abs(eigvals))
print('\n')

# Spectral decomposition of H from section 3:
Hspectral = eigvecs_eigh @ np.diag(eigvals_eigh) @ np.conjugate(eigvecs_eigh.T)
print_matrix(Hspectral)
print('\n')
print_matrix(H)

True
[ 0.71 + 0.71i , 0.71 - 0.71i ]
[ 1.0 + 0.0i , 1.0 + 0.0i ]


[ 2.0 + 0.0i , 1.0 - 1.0i , 0.0 + -0.0i ]
[ 1.0 + 1.0i , 3.0 + 0.0i , 2.0 + 0.0i ]
[ 0.0 + 0.0i , 2.0 + 0.0i , 1.0 + 0.0i ]


[ 2.0 + 0.0i , 1.0 - 1.0i , 0.0 + 0.0i ]
[ 1.0 + 1.0i , 3.0 + 0.0i , 2.0 + 0.0i ]
[ 0.0 + 0.0i , 2.0 + 0.0i , 1.0 + 0.0i ]


## 6. Matrix exponentials via spectral decomposition

In [7]:
# Compute exp(H)
ED = [[np.exp(eigvals_eigh[0]), 0, 0],
      [0, np.exp(eigvals_eigh[1]), 0],
      [0, 0, np.exp(eigvals_eigh[2])]]
EH =  eigvecs_eigh @ ED @ np.conjugate(eigvecs_eigh.T)
print_matrix(EH)

print('\n')

print_matrix(expm(H)) # Using built-in function

[ 24.11 + 0.0i , 27.14 - 27.14i , 13.18 - 13.18i ]
[ 27.14 + 27.14i , 77.62 + 0.0i , 41.11 + 0.0i ]
[ 13.18 + 13.18i , 41.11 + 0.0i , 23.33 + 0.0i ]


[ 24.11 + 0.0i , 27.14 - 27.14i , 13.18 - 13.18i ]
[ 27.14 + 27.14i , 77.62 + 0.0i , 41.11 + 0.0i ]
[ 13.18 + 13.18i , 41.11 + 0.0i , 23.33 + 0.0i ]
